# 7.3 Clusters in graphs — and whether they are there

[07.2](07.2-graph-properties.ipynb) measured one node at a time. This notebook asks the group
question: **does this network fall apart into communities, and how would you know if it did
not?**

That second half is the whole notebook. Every clustering algorithm here returns clusters. Run
`louvain_communities` on a graph with no structure in it whatsoever and you get communities, with
sizes and colours and a modularity score that looks respectable. There are no error bars and
nothing refuses. So the technique is not "cluster the graph"; the technique is **cluster the
graph, then establish that the clusters are not what any graph of that shape would have given
you.** By the end there is a verdict on the IRC chat network, and it is not a flattering one.

The worked example is Zachary's karate club, for one reason: the true answer was recorded
independently of the data. Barabási's [*Network Science*](http://networksciencebook.com/) chapter
[9](http://networksciencebook.com/chapter/9) is the reference — modularity and its limits in 9.4,
testing communities in 9.6, and the Louvain algorithm in advanced topic 9.C.

In [ ]:
import itertools

import networkx as nx
import numpy as np
import pandas as pd
from goad_toolkit.visualizer import HeatmapPlot, LinePlot, PlotSettings
from scipy.stats import kruskal
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import adjusted_rand_score

from scripts.graphs import (
    GraphPlot,
    HourProfile,
    MentionEdges,
    best_partition,
    cosine_edges,
    giant_component,
    labels,
    modularity_check,
    to_graph,
)
from scripts.pipelines import BuildTimestamp, build_irc_pipeline
from wa_analyzer.data import load_showcase
from wa_analyzer.network_analysis import Config, GraphBuilder

## 7.3.1 A split that actually happened

Wayne Zachary watched a university karate club for three years in the 1970s and recorded who
spent time with whom outside the dojo. During the study the administrator ("Mr. Hi") and the
instructor fell out, and the club split in two. Zachary wrote down which half each of the 34
members joined.

So this graph comes with a label column that was *not* derived from the edges. That is rare and
it is why the dataset is famous: it lets you ask whether an algorithm that never saw the labels
recovers them.

In [ ]:
karate = nx.karate_club_graph()
truth = np.array([0 if karate.nodes[n]["club"] == "Mr. Hi" else 1 for n in karate.nodes()])
pos = nx.spring_layout(karate, seed=46)

club = PlotSettings(figsize=(11, 5), max_cols=2, title="Zachary's karate club, 34 members",
                    subplot_titles=["the graph", "the split that happened"])
host = GraphPlot(club)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(GraphPlot(club), axes[0], data=karate, pos=pos, node_size=180, with_labels=True)
_ = host.plot_on_axes(GraphPlot(club), axes[1], data=karate, pos=pos, node_size=180,
                      with_labels=True, color=["#4c72b0" if t == 0 else "#c44e52" for t in truth])
print(f"{karate.number_of_nodes()} members, {karate.number_of_edges()} ties, "
      f"{int((truth == 0).sum())} went with Mr. Hi and {int((truth == 1).sum())} with the officer")

## 7.3.2 The Laplacian, and why its *smallest* eigenvalues

Two matrices describe a graph completely (Barabási 2.4). The **adjacency matrix** $A$ has
$A[i,j] = 1$ when $i$ and $j$ are connected. The **degree matrix** $D$ is diagonal, holding each
node's degree. The **Laplacian** is their difference:

$$L = D - A$$

Every row of $L$ sums to zero — your degree on the diagonal, minus one for each neighbour. That
is not a coincidence, it is the reason the whole method works: it means the vector of all ones is
an eigenvector with eigenvalue 0, always, for every graph.

In PCA (06.1) the *largest* eigenvalues mattered, because they carried the most variance. Here it
is the smallest, and the reason is worth holding on to. For any vector $x$ over the nodes,

$$x^{\top} L x = \sum_{(i,j) \in E} (x_i - x_j)^2$$

— the total disagreement between connected nodes. An eigenvector with a *small* eigenvalue is a
way of assigning numbers to people that neighbours mostly agree on. The smallest is the trivial
one (give everybody the same number, disagreement zero). The **second smallest** is the smallest
non-trivial one: the way to split the numbers so that as few edges as possible are cut. It has a
name, the **Fiedler vector**, and its eigenvalue measures how easy the graph is to cut in two.

The count of zero eigenvalues is the number of connected components — a disconnected graph can
assign a different constant to each piece for free.

In [ ]:
A = nx.to_numpy_array(karate)
D = np.diag(A.sum(axis=1))
L = D - A

values, vectors = np.linalg.eigh(L)
order = np.argsort(values)
values, vectors = values[order], vectors[:, order]
print(f"L is {L.shape}, rows sum to {L.sum(axis=1).max():.1e}")
print(f"the ten smallest eigenvalues: {np.round(values[:10], 3)}")
print(f"eigenvalues equal to zero: {int(np.isclose(values, 0, atol=1e-9).sum())} "
      f"-> {nx.number_connected_components(karate)} connected component")
print(f"the gap after the Fiedler value: {values[2] / values[1]:.2f}x")

spectrum = PlotSettings(figsize=(7, 4), title="Karate club: the Laplacian spectrum",
                        xlabel="eigenvalue index", ylabel="eigenvalue")
fig, ax = LinePlot(spectrum).plot(data=pd.DataFrame({"i": range(len(values)), "v": values}),
                                  x="i", y="v", marker="o", color="#666666")
_ = ax.axvline(1, color="#c44e52", linestyle="--")

One zero, then 1.19, then 2.39: the second eigenvalue is a factor two below the third. **That gap
is the only thing in this notebook that tells you how many clusters to look for**, and here it
says two. Hold on to how rare that will turn out to be.

## 7.3.3 The Fiedler vector, coloured by sign

The second eigenvector gives every member a number. Colour by whether that number is above or
below zero and compare against the split Zachary recorded.

In [ ]:
fiedler = vectors[:, 1]
side = (fiedler > 0).astype(int)
agreement = max((side == truth).mean(), 1 - (side == truth).mean())
wrong = [n for n, s, t in zip(karate.nodes(), side, truth)
         if (s == t) != ((side == truth).mean() > 0.5)]
print(f"Fiedler split: {int(side.sum())} vs {int((1 - side).sum())} members")
print(f"agreement with the recorded split: {agreement:.1%}  (ARI {adjusted_rand_score(truth, side):.3f})")
print(f"members on the wrong side: {wrong}")

split = PlotSettings(figsize=(11, 5), max_cols=2, title="The second eigenvector against the record",
                     subplot_titles=["sign of the Fiedler vector", "the split that happened"])
host = GraphPlot(split)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(GraphPlot(split), axes[0], data=karate, pos=pos, node_size=180, with_labels=True,
                  color=["#4c72b0" if s == 0 else "#c44e52" for s in side])
_ = host.plot_on_axes(GraphPlot(split), axes[1], data=karate, pos=pos, node_size=180,
                      with_labels=True, color=["#4c72b0" if t == 0 else "#c44e52" for t in truth])

**Thirty-three of thirty-four.** An eigenvector of a matrix built from nothing but who trained
with whom reproduces a social rupture that happened two years later, and misplaces one person.

That person is member 8, and the miss is the interesting part rather than a rounding error.
Member 8 has ties on both sides; Zachary's own account records that he joined the officer's club
for a reason outside the graph — he was three weeks from a black belt under Mr. Hi and would have
lost the rank by switching. **The one node the mathematics gets wrong is the one whose decision
was made on information the network does not contain.** That is the honest shape of a good
result, and it is worth remembering when you have no labels to be wrong against.

## 7.3.4 More clusters, and the price of asking for them

The sign of one eigenvector splits a graph in two. For $k$ clusters the standard move is to hand
the first few eigenvectors to KMeans and treat each node's eigenvector values as its coordinates
— the graph embedded in a space where connected nodes sit close (06.1's move, on a different
matrix).

In [ ]:
rows = []
for k in (2, 3, 4, 5):
    embedding = vectors[:, 1:k]
    assignment = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(embedding)
    rows.append({"k": k, "eigenvectors used": f"2..{k}",
                 "sizes": np.bincount(assignment).tolist(),
                 "ARI vs the record": adjusted_rand_score(truth, assignment)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

At $k=2$ KMeans on the Fiedler vector alone reproduces the sign split exactly, ARI 0.88. Every
larger $k$ is worse — 0.68, 0.56, 0.59 — and at $k=4$ and $k=5$ one of the "communities" is a
single member, which is what an algorithm asked for four groups does when the graph has two.

Nothing in the output says which row to believe. The eigenvalue plot did, once: the gap after
the second eigenvalue said two, and two is the row that works. **The number of clusters is not a
parameter you tune until the picture looks good** — it is a claim, and the spectrum is the only
evidence for it that the graph itself provides. When there is no gap, there is no evidence.

## 7.3.5 Modularity, and why it is not a score

Louvain (Barabási 9.12) takes a different route: it looks for a partition that maximises
**modularity** $Q$ — how many more edges fall inside communities than would if the same nodes
with the same degrees were wired at random (Barabási 9.4).

In [ ]:
partition = best_partition(karate)
karate_labels = labels(karate, partition)
print(f"louvain: Q = {nx.community.modularity(karate, partition, weight=None):.3f}, "
      f"{len(partition)} communities, sizes {sorted((len(c) for c in partition), reverse=True)}")
print(f"ARI against the recorded split: "
      f"{adjusted_rand_score(truth, karate_labels.reindex(karate.nodes())):.3f}")

coarse = best_partition(karate, resolution=0.6)
coarse_labels = labels(karate, coarse)
print(f"\nlouvain at resolution 0.6: {len(coarse)} communities, "
      f"sizes {sorted((len(c) for c in coarse), reverse=True)}")
print(f"ARI against the recorded split: "
      f"{adjusted_rand_score(truth, coarse_labels.reindex(karate.nodes())):.3f}")

Left alone, louvain splits the club into **four** communities, not two, and its agreement with
the recorded split is 0.47 — it has cut both real halves in half. This is modularity's
**resolution limit** (Barabási 9.4, "Limits of Modularity"): maximising $Q$ has a built-in
preferred community size that depends on the size of the whole graph, so it will merge genuinely
separate small groups in a large network and subdivide large ones in a small network. Turn the
resolution down to 0.6 and you get two communities of seventeen and ARI 0.77 — most of the right
answer, from a parameter chosen because we already knew the right answer.

**That is the trap this whole notebook is about.** The four-community answer and the
two-community answer are both louvain output, both have a defensible $Q$, and only one of them
matches reality. Without the label column there is nothing in the run that prefers either.

## 7.3.6 The number that makes modularity mean something

$Q$ has no natural zero. A graph wired at random still has some partition that scores well,
because in any finite random graph some regions happen to be denser than others — and the
sparser the graph, the more room there is for that accident. So $Q = 0.4$ is not a result. The
result is $Q$ **compared against the same measurement on a graph that has the same degrees and
nothing else** (Barabási 9.6).

`modularity_check` builds that comparison: twenty copies of the graph with the edges rewired by
repeated double swaps, which keeps every node's degree exactly, clusters each one the same way,
and reports how many standard deviations the real graph sits above them.

In [ ]:
pipeline = build_irc_pipeline()
pipeline.add(BuildTimestamp)
irc = pipeline.apply(load_showcase("ubuntu_irc"))
chat = irc[(irc.channel == "#ubuntu-uk") & (irc.date.dt.year == 2015)].copy()

mentions = giant_component(to_graph(MentionEdges()(chat), nodes=chat.author.unique()))
builder = GraphBuilder(Config(time_col="timestamp", node_col="author", seconds=600, datafile=None))
nearby = giant_component(builder.build(chat, edge_seconds=600))

checks = {
    "karate club": modularity_check(karate),
    "florentine families": modularity_check(nx.florentine_families_graph()),
    "irc mentions": modularity_check(mentions),
    "irc nearby, 600s": modularity_check(nearby),
}
sizes = {"karate club": karate, "florentine families": nx.florentine_families_graph(),
         "irc mentions": mentions, "irc nearby, 600s": nearby}
table = pd.DataFrame([
    {"graph": name, "nodes": sizes[name].number_of_nodes(), "edges": sizes[name].number_of_edges(),
     "Q": check.q, "Q rewired": check.null_mean, "spread": check.null_std,
     "z": check.z, "k": check.k}
    for name, check in checks.items()
])
print(table.round(3).to_string(index=False))

Read the `Q` column and the `z` column against each other, because they rank the four graphs in
almost opposite orders.

**The Florentine families score $Q = 0.40$ — and $z = +0.6$.** Rewire those twenty marriages at
random, keeping each family's number of in-laws, and you get a modularity just as high. The
fifteen-node graph that [07.2](07.2-graph-properties.ipynb) read so confidently for centrality
has no community structure that survives the check. It is too small for the question: with twenty
edges, *any* partition into four groups leaves a lot of edges inside by luck.

**The karate club scores $Q = 0.42$ — barely higher — and $z = +11$.** Almost the same
modularity, an entirely different verdict. This is the graph with real communities in it.

**The IRC mention graph scores $Q = 0.22$ and $z = +2.6$.** Two and a half standard deviations is
not nothing, and it is also not much: the mention network has slightly more structure than a
random network with the same degree sequence, and slightly is the honest word. Almost all of its
modularity is the modularity any graph of that shape would have had.

The lesson generalises past graphs. **A statistic with no natural zero needs a null, and the null
has to hold constant everything you are not asking about** — here, every node's degree, because
a heavy-tailed degree sequence produces high modularity all by itself. Reporting $Q$ alone is
like reporting an $R^2$ with no idea how many predictors were fitted.

## 7.3.7 The karate recipe, applied where it does not work

Now the same spectral machinery on the mention graph. 256 people, no labels, no known answer.

In [ ]:
A_irc = nx.to_numpy_array(mentions)
L_irc = np.diag(A_irc.sum(axis=1)) - A_irc
irc_values, irc_vectors = np.linalg.eigh(L_irc)
order = np.argsort(irc_values)
irc_values, irc_vectors = irc_values[order], irc_vectors[:, order]

irc_fiedler = irc_vectors[:, 1]
nodes = list(mentions.nodes())
degrees = np.array([mentions.degree(n) for n in nodes])
loudest = np.argsort(-np.abs(irc_fiedler))[:6]

print(f"the eight smallest eigenvalues: {np.round(irc_values[:8], 4)}")
print(f"gap after the Fiedler value: {irc_values[2] / irc_values[1]:.2f}x")
print(f"sign split: {int((irc_fiedler > 0).sum())} vs {int((irc_fiedler <= 0).sum())} nodes\n")
print("the six nodes with the largest |Fiedler value|, and their degrees:")
for i in loudest:
    print(f"  {nodes[i]:20s} value {irc_fiedler[i]:+.3f}   degree {degrees[i]}")
print(f"\nmedian degree in this graph: {np.median(degrees):.0f}")

**The Fiedler vector cuts two people off the graph and leaves the other 254 together.** It is
not wrong: those really are the two cheapest edges to cut, and the vector is doing exactly what
§7.3.2 said it would. It is *useless*, and the reason is in the degree column — every node it
picks out has degree one or two.

In the karate club everybody has a comparable number of ties — 1 to 17, on 34 members — so the
cheapest cut really is the cut between the two halves. In the mention graph the degrees run from
1 to 116 on a median of 3, the heavy tail [07.2](07.2-graph-properties.ipynb) §7.2.8 measured,
and the cheapest cut is always "detach a person with one edge". Minimising the cut without
controlling for the size of the pieces finds the smallest possible piece, every time.

The standard fix is the **normalised Laplacian**, $L_{sym} = I - D^{-1/2} A D^{-1/2}$, which
divides the cut by the volume of each side and so refuses to call one node a community. Applying
it here is worth doing because of what it exposes about the spectrum.

In [ ]:
inv_sqrt = np.diag(1 / np.sqrt(np.maximum(A_irc.sum(axis=1), 1e-12)))
L_sym = np.eye(len(nodes)) - inv_sqrt @ A_irc @ inv_sqrt
sym_values, sym_vectors = np.linalg.eigh(L_sym)
order = np.argsort(sym_values)
sym_values, sym_vectors = sym_values[order], sym_vectors[:, order]
print(f"normalised spectrum, ten smallest: {np.round(sym_values[:10], 4)}")
print(f"successive ratios: {np.round(sym_values[2:10] / sym_values[1:9], 3)}")

louvain_labels = labels(mentions, best_partition(mentions, weight="weight"))
rows = []
for k in (2, 3, 4, 5, 6):
    embedding = sym_vectors[:, 1:k + 1]
    embedding = embedding / np.maximum(np.linalg.norm(embedding, axis=1, keepdims=True), 1e-12)
    assignment = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(embedding)
    communities = [{nodes[i] for i in np.where(assignment == j)[0]} for j in range(k)]
    rows.append({"k": k, "sizes": sorted(np.bincount(assignment).tolist(), reverse=True),
                 "Q": nx.community.modularity(mentions, [c for c in communities if c], weight=None),
                 "ARI vs louvain": adjusted_rand_score(assignment, louvain_labels.reindex(nodes))})
print()
print(pd.DataFrame(rows).round(3).to_string(index=False))

The normalised Laplacian gives balanced clusters and a modularity of about 0.16 at every $k$ — and
**the spectrum has no gap anywhere**: 0.42, 0.47, 0.57, 0.59, 0.63, each ratio within a few
percent of the last. In the karate club the gap was a factor of two and it told us $k = 2$. Here
there is no $k$ the graph prefers, and $k$ is now purely something we chose. Nor do the methods
agree with each other: spectral at $k=3$ overlaps louvain at ARI 0.67, its best showing, which
still leaves a third of the pairwise decisions different between two reasonable algorithms
running on one graph.

## 7.3.8 Stability: the same rule, twice

A cluster that is a property of these people should survive being computed slightly differently.
Four perturbations, none of them exotic — a different random seed, a different resolution, a
different half of the year, a different edge rule — and the same measurement each time: the
**adjusted Rand index**, the share of pairs that two clusterings agree about, corrected for the
agreement you would get by chance. 1.0 is identical, 0.0 is no better than random.

In [ ]:
seed_labels = [labels(mentions, nx.community.louvain_communities(mentions, weight="weight", seed=s))
               .reindex(nodes).to_numpy() for s in range(10)]
seed_ari = [adjusted_rand_score(a, b) for a, b in itertools.combinations(seed_labels, 2)]
print(f"across 10 louvain seeds:      ARI {np.mean(seed_ari):.3f} +- {np.std(seed_ari):.3f}")

base = labels(mentions, best_partition(mentions, weight="weight")).reindex(nodes).to_numpy()
for resolution in (0.75, 1.25, 1.5):
    other = labels(mentions, best_partition(mentions, weight="weight", resolution=resolution))
    print(f"resolution {resolution} vs 1.0:      "
          f"ARI {adjusted_rand_score(base, other.reindex(nodes)):.3f}")

In [ ]:
half = pd.Timestamp("2015-07-01")
first = giant_component(to_graph(MentionEdges()(chat[chat.timestamp < half])))
second = giant_component(to_graph(MentionEdges()(chat[chat.timestamp >= half])))
first_labels = labels(first, best_partition(first, weight="weight"))
second_labels = labels(second, best_partition(second, weight="weight"))
survivors = sorted(set(first.nodes()) & set(second.nodes()))
print(f"first half: {first.number_of_nodes()} people, second half: {second.number_of_nodes()}, "
      f"in both: {len(survivors)}")
print(f"January-June vs July-December:  ARI "
      f"{adjusted_rand_score(first_labels.reindex(survivors), second_labels.reindex(survivors)):.3f}")

nearby_labels = labels(nearby, best_partition(nearby, weight="weight"))
shared = sorted(set(nearby.nodes()) & set(mentions.nodes()))
print(f"mention rule vs nearby rule:    ARI "
      f"{adjusted_rand_score(labels(mentions, best_partition(mentions, weight='weight')).reindex(shared), nearby_labels.reindex(shared)):.3f}")

The seeds agree with each other well enough — louvain is randomised but it is not chaotic. Every
other line is a failure.

**The same rule on the same channel, six months apart, agrees at ARI 0.08.** Of the eighty people
who were around in both halves of 2015, knowing who clustered together in January tells you
essentially nothing about who clusters together in August — 0.08 is a hair above the zero that
means "no better than shuffling the labels". Whatever the algorithm found, it is not a property
of these people that persists for six months.

**The mention rule and the nearby rule agree at ARI 0.18** on the 255 people both graphs contain.
07.1 showed the two rules produce quite similar *degrees* — spearman 0.90. Their *communities*
have almost nothing in common. Aggregate agreement between two rules is not agreement about
structure.

Turning the resolution knob is a third failure of the same kind, and one worth naming precisely:
it is not that the clusters get finer, which would be fine. It is that they get *rearranged*.

## 7.3.9 If a cluster is real, it should show up in something else

There are no labels, so the check is against columns the clustering never saw. Each cluster
either differs from the others in some observable way or it does not, and "does not" is an
answer.

In [ ]:
observed = pd.DataFrame({"cluster": base}, index=nodes)
observed["messages"] = chat.author.value_counts().reindex(nodes)
observed["median_hour"] = chat.groupby("author").hh.median().reindex(nodes)
observed["url_rate"] = chat.groupby("author").has_url.mean().reindex(nodes)
observed["questions"] = chat.groupby("author").n_question.mean().reindex(nodes)

print(observed.groupby("cluster").agg(people=("messages", "size"), median_messages=("messages", "median"),
                                      median_hour=("median_hour", "median"),
                                      url_rate=("url_rate", "mean")).round(3).to_string())
print()
for column in ["messages", "median_hour", "url_rate", "questions"]:
    groups = [g[column].dropna().to_numpy() for _, g in observed.groupby("cluster") if len(g) >= 3]
    grand = observed[column].mean()
    total = ((observed[column] - grand) ** 2).sum()
    between = sum(len(g) * (g[column].mean() - grand) ** 2 for _, g in observed.groupby("cluster"))
    print(f"{column:13s} kruskal p = {kruskal(*groups).pvalue:8.1e}   "
          f"variance explained by cluster: {between / total:5.1%}")

Activity, link-posting and question-asking are **flat across the clusters**: the algorithm did
not separate the chatty from the quiet or the helpers from the askers, and the three p-values are
0.36, 0.38 and 0.71. One column is not flat: median hour of the day differs strongly
($p \approx 10^{-9}$) and accounts for 14% of the variance between people.

So the communities are, weakly, groups of people who are awake at the same time — which raises
an obvious question. If time of day is what the mention graph is faintly tracking, what happens
if you build a graph out of time of day directly?

## 7.3.10 Two more edge rules, and a one-line rejection

Before that, the rule most people reach for first: **connect people who use the same
vocabulary.** Paste each author's messages into one document, tf-idf, cosine similarity, keep the
most similar pairs. It is a projection of a bipartite author–term network (Barabási 2.7), which is
the honest way to describe it, and it sounds like it should find the people who talk about the
same things.

In [ ]:
counts = chat.author.value_counts()
regulars = counts[counts >= 50].index
documents = chat[chat.author.isin(regulars)].groupby("author").message.apply(" ".join)

tfidf = TfidfVectorizer(min_df=5, max_df=0.5, sublinear_tf=True)
matrix = tfidf.fit_transform(documents.to_numpy())
matrix = matrix / np.sqrt(matrix.multiply(matrix).sum(axis=1))
similarity = (matrix @ matrix.T).toarray()
np.fill_diagonal(similarity, 0.0)

mean_similarity = similarity.sum(axis=1) / (len(documents) - 1)
sent = counts.reindex(documents.index).to_numpy()
print(f"{len(documents)} authors with 50+ messages, {len(tfidf.vocabulary_):,} terms")
print(f"spearman(messages sent, mean cosine to everyone else) = "
      f"{pd.Series(sent).corr(pd.Series(mean_similarity), method='spearman'):+.2f}")
print(f"\nmost similar to everyone: {list(documents.index[np.argsort(-mean_similarity)[:5]])}")
print(f"most messages sent:       {list(counts.reindex(documents.index).nlargest(5).index)}")

**Spearman 0.96, and four of the five names are the same in both lists.** The "shared vocabulary" graph is
an activity graph. Someone who wrote fourteen thousand words used most of the vocabulary at some
point, so they are similar to everybody; someone who wrote four hundred used a handful of terms
and is similar to nobody. Cosine on tf-idf is supposed to be length-invariant and it is, per
document — but a long document *covers more of the vocabulary*, and coverage is what drives the
overlap.

Cluster that graph and you will get clean, stable communities. They will be activity tiers, and
every sentence you write about them ("this community discusses networking issues") will be a
sentence about how much its members type. **The check cost one line and it happened before any
clustering ran** — which is the only place it does any good.

## 7.3.11 The graph that does cluster

So: time of day, directly. `HourProfile` gives each author 24 numbers, the share of their
messages sent in each hour. The obvious objection is that this is trivial — everybody sleeps at
night, so everybody's profile looks alike and the clusters will be "awake" and "asleep". The
`normalise` step is what tests that objection: divide each hour by the channel's own share of
traffic in that hour, so the columns say *unusual for this channel* rather than *busy*.
`cosine_edges` then connects the most similar tenth of all pairs.

In [ ]:
profiles = HourProfile()(chat[chat.author.isin(regulars)])
edges = cosine_edges(profiles, quantile=0.90)
copresence = giant_component(to_graph(edges.drop(columns="threshold")))
print(f"cosine threshold at the 90th percentile: {edges.threshold.iloc[0]:.3f}")
print(f"co-presence graph: {copresence.number_of_nodes()} nodes, "
      f"{copresence.number_of_edges()} edges\n")

raw_profiles = HourProfile(normalise=False)(chat[chat.author.isin(regulars)])
for label, frame in [("raw hour shares", raw_profiles), ("normalised per hour", profiles)]:
    graph = giant_component(to_graph(cosine_edges(frame, quantile=0.90).drop(columns="threshold")))
    print(f"{label:22s} {modularity_check(graph)}")

**The objection does not survive contact with the check.** Dividing out the channel's own
day–night curve makes the structure *stronger*, not weaker: $Q$ goes up and the graph sits forty
standard deviations above its rewired null — by far the strongest result in this notebook, and
an order of magnitude above the mention graph's $z = +2.6$.

The clusters are readable, which is the part that matters. Take the partition and plot each
cluster's average raw hour profile.

In [ ]:
groups = labels(copresence, best_partition(copresence))
members = list(copresence.nodes())
shares = raw_profiles.reindex(members)
mean_profile = (shares.groupby(groups.reindex(members)).mean() * 100)
mean_profile.index = [f"cluster {i} (n={int((groups == i).sum())})" for i in mean_profile.index]
print(f"cluster sizes:               {groups.value_counts().sort_index().tolist()}")
print(f"peak hour per cluster (UTC): {mean_profile.to_numpy().argmax(axis=1).tolist()}")

heat = PlotSettings(figsize=(11, 3.5), title="Co-presence clusters are hours of the day",
                    xlabel="hour of day (UTC)", ylabel="")
fig, ax = HeatmapPlot(heat).plot(data=mean_profile, cmap="Blues", cbar_kws={"label": "% of own messages"})

Four bands: a morning cluster of 27 people peaking around 10:00 UTC, an afternoon one of 19 at
16:00, an evening one of 20 at 22:00, and an early-hours group of 14 at 05:00. The fifth cluster
is three people and is noise. These are working hours and time zones, and the reading is
defensible because the heatmap is the evidence — drawn from the column the clustering was built
on, which is exactly the reason it is not much of a discovery.

And now the caveat that decides what the whole thing is worth.

In [ ]:
print(f"seed stability, 10 louvain runs: ARI "
      f"{np.mean([adjusted_rand_score(labels(copresence, nx.community.louvain_communities(copresence, seed=a)).reindex(members), labels(copresence, nx.community.louvain_communities(copresence, seed=b)).reindex(members)) for a, b in itertools.combinations(range(6), 2)]):.3f}")
for quantile in (0.85, 0.95):
    other = giant_component(to_graph(cosine_edges(profiles, quantile=quantile).drop(columns="threshold")))
    common = sorted(set(other.nodes()) & set(members))
    print(f"threshold p{quantile * 100:.0f} vs p90: {other.number_of_nodes():3d} nodes, ARI "
          f"{adjusted_rand_score(labels(other, best_partition(other)).reindex(common), groups.reindex(common)):.3f} "
          f"on the {len(common)} shared")

mention_groups = labels(mentions, best_partition(mentions, weight="weight"))
overlap = sorted(set(members) & set(mentions.nodes()))
print(f"\nco-presence clusters vs mention clusters: ARI "
      f"{adjusted_rand_score(groups.reindex(overlap), mention_groups.reindex(overlap)):.3f} "
      f"on the {len(overlap)} people in both")

The co-presence clustering is stable against the seed and against a threshold moved from the 90th
percentile to the 85th (ARI 0.87). At the 95th it falls apart, and the reason is visible in the
node count: that graph keeps 34 of the 83 people, and a clustering of a different set of people
is not a perturbation of this one. Reporting a threshold sweep means reporting which end of it
stopped being the same question.

The last line is the one to sit with. **The co-presence clusters and the mention clusters agree
at ARI 0.18.** So the graph with the strongest, most defensible community structure in this
notebook is telling you about *clocks*, and it disagrees with the graph that is about
*conversations*. Nobody in the 22:00 cluster needs to have exchanged a single word with anybody
else in it.

That is a real finding and it is not the finding anyone wanted. "I can recover four time zones
from an IRC log" is defensible and slightly dull; "I found the four social groups in this
channel" is what people write, and it is not what happened.

## 7.3.12 The verdict

Laid out plainly, because a null result is only worth anything if it is stated as sharply as a
positive one would have been:

| edge rule | $Q$ | $z$ vs rewired | stable across halves of the year? | tracks anything observable? |
|---|---|---|---|---|
| who was addressed (mention) | 0.22 | +2.6 | no, ARI 0.08 | hour of day, 14% of variance |
| who was nearby (10 min) | 0.16 | +12 | — | agrees with mentions at ARI 0.18 |
| shared vocabulary (tf-idf) | — | — | — | activity, spearman 0.96 — rejected before clustering |
| co-presence (hour profile) | 0.48 | +46 | partly | hour of day, by construction |

**The `#ubuntu-uk` mention network does not have defensible communities.** Its modularity is
barely above a graph with the same degrees wired at random; its clusters do not survive splitting
the year in two; they do not survive changing the edge rule; and the only thing that distinguishes
them is a variable another graph measures far better. Every one of those is a fact you can show
someone.

What you may say: *"Louvain finds nine communities in the mention graph at $Q = 0.22$. Twenty
rewirings that keep every degree give $Q = 0.21 \pm 0.003$, so the real graph is 2.6 standard
deviations above chance. The partition does not replicate between the first and second half of
2015 (ARI 0.08) and does not agree with the nearby rule (ARI 0.18). The only observable that
separates the clusters is time of day, at 14% of the variance. I do not think these are social
groups."*

What you may not say: any sentence naming a cluster.

## 7.3.13 Your turn

Your own chat is small, so louvain will return communities immediately and they will look
convincing. Run the null before you look at them.

```python
from scripts.graphs import best_partition, giant_component, labels, modularity_check
from wa_analyzer.data import load_own_chat
from wa_analyzer.network_analysis import Config, GraphBuilder

own = load_own_chat()
own["timestamp"] = pd.to_datetime(own["timestamp"])
graph = giant_component(GraphBuilder(Config(time_col="timestamp", node_col="author",
                                            seconds=600, datafile=None)).build(own, edge_seconds=600))
print(modularity_check(graph))            # Q, the rewired Q, and z

first = own[own.timestamp < own.timestamp.median()]
second = own[own.timestamp >= own.timestamp.median()]
# ...build a graph from each, cluster both, and compare with adjusted_rand_score
```

> **Your turn.**
>
> 1. Run `modularity_check` on your own chat graph. Report $Q$, the rewired $Q$ and $z$ together
>    — never $Q$ alone. In a group of eight the graph is nearly complete and $z$ will be small;
>    say so.
> 2. Split your chat in half by time, cluster each half, and report the ARI between them on the
>    people present in both. This is the single most convincing number you can produce about a
>    clustering, in either direction.
> 3. Take your clusters and check them against a column the clustering never saw: messages sent,
>    median hour, mean message length, share of messages with a link. If nothing separates them,
>    write that down. It is the result.
> 4. If your chat has a structure you *know* about — a family half and a work half — use it as
>    karate-club labels and report the ARI. You are one of very few people in this course with a
>    real label column; use it.

## 7.3.14 What to write down

1. **$Q$ and the null together**, always, plus how the null was built.
2. **How many clusters, and why that number.** An eigengap, a resolution you can justify, or an
   admission that you picked it.
3. **One stability number** — seeds, threshold, or a split of the data, ideally the last.
4. **What the clusters track that the clustering never saw** — or the fact that nothing does.
5. **The sentence you are entitled to.** Write it out and check every clause against a number
   above it.

---

**Where this goes next.** Nothing, in this course — [07.1](07.1-social_graphs.ipynb) said the
tools stop at lesson 7. What does not stop is the shape of all three notebooks in this lesson,
which is the shape of the whole course: a technique that runs, a decision inside it that the
technique cannot make, and a check that tells you whether the output survived the decision. The
edge rule in 07.1, the layout and the choice of centrality in 07.2, the number of clusters and
the null model here.

You will be asked to defend one of those choices, and "a null result, bounded" is a defence. "The
algorithm returned four communities" is not.